# モジュラモノリス（Modular Monolith）

単一のデプロイ単位（モノリス）を維持しながら、内部を疎結合なモジュールに明確に分割するアーキテクチャスタイル。
[マイクロサービス](../distributed/microservices.ipynb) ほどの運用コストをかけずに、モノリスが陥りがちな
「密結合の塊（Big Ball of Mud）」を避けることを狙う。

## 構造

モジュールの境界は、[DDDの境界づけられたコンテキスト](../../design/domain_driven_design/strategic_design.ipynb) に
沿って引かれることが多い。各モジュールは独立したパッケージ（ディレクトリ）として構成され、モジュール間の呼び出しは
公開されたインターフェース経由に限定し、内部実装への直接アクセスを禁止する。

```
src/
├── ordering/          # 注文モジュール
│   ├── domain/
│   ├── application/
│   └── public_api.py  # 他モジュールに公開するインターフェース
├── inventory/          # 在庫モジュール
│   ├── domain/
│   ├── application/
│   └── public_api.py
└── shared_kernel/       # 共有カーネル（最小限に留める）
```

「垂直スライス・アーキテクチャ（Vertical Slice Architecture）」も近い発想で、レイヤー（技術的関心事）ではなく
機能・ユースケース単位でコードをまとめる編成方法として語られることが多い。

## マイクロサービスとの比較

| | モジュラモノリス | マイクロサービス |
| --- | --- | --- |
| デプロイ単位 | 1つ | サービスごと |
| モジュール間通信 | プロセス内呼び出し（速い、トランザクションを跨ぎやすい） | ネットワーク越し（[Saga](../distributed/saga_distributed_transaction.ipynb) 等が必要） |
| 運用コスト | 低い（単一デプロイ・単一DB） | 高い（分散システムの複雑さ、[観測性](../quality_attributes.ipynb)投資が必須） |
| 境界の緩みやすさ | モジュール間の呼び出しがコンパイラでは強制しにくく、規律が必要 | プロセス境界が物理的に強制する |
| チームの独立性 | デプロイは共有される | サービスごとに独立してデプロイ可能 |

多くのプロジェクトにとって、最初からマイクロサービスに分割するよりも、モジュラモノリスとして境界を明確にしておき、
実際に必要になった時点で特定のモジュールだけをサービスとして切り出す方が、リスクとコストのバランスが良いことが多い。